In [1]:
import subprocess, os

PKGS = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"

# Install everything from wheel cache (including unsloth)
subprocess.run(
    f"pip install -q --no-index --find-links {PKGS} "
    "unsloth unsloth_zoo trl peft transformers datasets accelerate bitsandbytes safetensors",
    shell=True, check=True
)

# mamba_ssm and causal_conv1d are in the parent dir, not /packages
subprocess.run(
    "pip install -q --no-deps "
    "/kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
    shell=True, check=True
)
subprocess.run(
    "pip install -q --no-deps "
    "/kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
    shell=True, check=True
)

subprocess.run("rm -rf /kaggle/tmp/*", shell=True)
print("Done")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.


Done


In [2]:
# ── Cell 2: Config ─────────────────────────────────────────────────────────
LORA_RANK    = 32
LORA_ALPHA   = 64
LORA_DROPOUT = 0.05

SFT_MAX_SEQ  = 3072
SFT_EPOCHS   = 1
SFT_BATCH    = 1
SFT_GRAD_ACCUM = 8
SFT_LR       = 2e-4

TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "up_proj", "down_proj", "in_proj", "out_proj",
]

MODEL_PATH   = None
SFT_DATA_DIR = "/kaggle/input/nemotron-sft-dataset/sft_hf_dataset"
SFT_OUT      = "/kaggle/working"

In [3]:
# ── Cell 3: Verify env ─────────────────────────────────────────────────────
import torch, mamba_ssm, causal_conv1d

cc = torch.cuda.get_device_capability(0)
print(f"GPU : {torch.cuda.get_device_name(0)}, sm_{cc[0]*10+cc[1]}")
print(f"torch={torch.__version__}, cuda={torch.version.cuda}")
print(f"mamba_ssm={mamba_ssm.__version__}, causal_conv1d={causal_conv1d.__version__}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

from causal_conv1d import causal_conv1d_fn
_x = torch.randn(1, 512, 32, device="cuda", dtype=torch.bfloat16) + 4e-3
_w = torch.randn(512, 4, device="cuda", dtype=torch.bfloat16)
causal_conv1d_fn(_x, _w, None, activation="silu")
print("causal_conv1d CUDA kernel: OK")

GPU : NVIDIA RTX PRO 6000 Blackwell Server Edition, sm_120
torch=2.10.0+cu128, cuda=12.8
mamba_ssm=2.3.1, causal_conv1d=1.6.1
VRAM: 102.0 GB
causal_conv1d CUDA kernel: OK


In [4]:
# ── Cell 4: Load model ─────────────────────────────────────────────────────
import unsloth
import kagglehub
from unsloth import FastLanguageModel

MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)
print(f"Model path: {MODEL_PATH}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=SFT_MAX_SEQ,
    load_in_4bit=False,
    load_in_8bit=False,
    full_finetuning=False,
    trust_remote_code=True,
    attn_implementation="eager",
    dtype=torch.bfloat16,
)
print("Model loaded.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/tmp/ipykernel_163/2040943152.py:2: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import unsloth
2026-05-14 07:19:32.081514: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778743172.301156     163 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778743172.378323     163 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778743172.957060     163 computation_placer.cc:177] computation placer already registered. Please 

🦥 Unsloth Zoo will now patch everything to make training faster!
Model path: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.3.17: Fast Nemotron_H patching. Transformers: 4.57.6.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards:   0%|          | 0/13 [00:00<?, ?it/s]

/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1 does not have a padding token! Will use pad_token = <SPECIAL_999>.
Model loaded.


In [5]:
# Cell 5: Apply LoRA - reduce targets to save memory
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "in_proj", "out_proj"],  # drop up_proj, down_proj (MoE experts)
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,   # dropout causes extra allocations
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

Unsloth: Making `model.base_model.model.backbone` require gradients


In [6]:
# ── Cell 6: Load & tokenize dataset ───────────────────────────────────────
from datasets import load_dataset

print("Loading SFT dataset...")
sft_ds = load_dataset(
    "json",
    data_files={
        "train": "/kaggle/input/datasets/tadityasasidhar/sft-hf-dataset/train.jsonl",
        "validation": "/kaggle/input/datasets/tadityasasidhar/sft-hf-dataset/validation.jsonl",
    }
)

def tokenize(example):
    out = tokenizer(
        example["text"],
        truncation=True,
        max_length=SFT_MAX_SEQ,
        padding=False,
    )
    out["labels"] = out["input_ids"].copy()
    return out

col = sft_ds["train"].column_names
sft_ds = sft_ds.map(
    tokenize,
    remove_columns=col,
    num_proc=2,
    desc="Tokenizing",
)
print("Tokenization done.")
print(sft_ds)

Loading SFT dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Tokenizing (num_proc=2):   0%|          | 0/6694 [00:00<?, ? examples/s]

Tokenizing (num_proc=2):   0%|          | 0/353 [00:00<?, ? examples/s]

Tokenization done.
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 6694
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 353
    })
})


In [7]:
# ── Cell 7: Train (raw loop + paged optimizer) ─────────────────────────────
import os, gc, math, torch
from torch.nn.utils import clip_grad_norm_
from datasets import load_dataset
import bitsandbytes as bnb

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# ── Config ─────────────────────────────────────────────────────────────────
MAX_SEQ_LEN   = 3072 
NUM_STEPS     = 300
BATCH_SIZE    = 32
MICRO_BATCH   = 1
LEARNING_RATE = 2e-4

# ── Load & tokenize ────────────────────────────────────────────────────────
raw_ds = load_dataset(
    "json",
    data_files={
        "train": "/kaggle/input/datasets/tadityasasidhar/sft-hf-dataset/train.jsonl",
    }
)["train"]

examples = []
for row in raw_ds:
    tokens = tokenizer(
        row["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
    )["input_ids"]
    if len(tokens) < 2:
        continue
    examples.append({
        "tokens": tokens[:-1],
        "targets": tokens[1:],
    })

print(f"Loaded {len(examples)} examples")

# ── Setup ──────────────────────────────────────────────────────────────────
FastLanguageModel.for_training(model)
gc.collect()
torch.cuda.empty_cache()

device    = next(model.parameters()).device
indices   = list(range(len(examples)))

max_steps = len(examples) // BATCH_SIZE
num_steps = min(NUM_STEPS, max_steps)
print(f"Training: {num_steps} steps, batch={BATCH_SIZE}, micro={MICRO_BATCH}, lr={LEARNING_RATE}")

# Paged AdamW — optimizer states live in CPU RAM, not VRAM
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = bnb.optim.PagedAdamW(
    trainable,
    lr=LEARNING_RATE,
    betas=(0.9, 0.95),
    eps=1e-8,
    weight_decay=0.0,
)

# ── Training loop ──────────────────────────────────────────────────────────
step = 0
for batch_start in range(0, len(indices), BATCH_SIZE):
    if step >= num_steps:
        break

    batch    = [examples[i] for i in indices[batch_start:batch_start + BATCH_SIZE]]
    n        = len(batch)
    n_accum  = math.ceil(n / MICRO_BATCH)
    loss_sum = 0.0

    for mb_start in range(0, n, MICRO_BATCH):
        mb      = batch[mb_start:mb_start + MICRO_BATCH]
        n_micro = len(mb)
        max_len = max(len(e["tokens"]) for e in mb)

        input_ids = torch.zeros(n_micro, max_len, dtype=torch.long, device=device)
        labels    = torch.full((n_micro, max_len), -100, dtype=torch.long, device=device)
        attn_mask = torch.zeros(n_micro, max_len, dtype=torch.long, device=device)

        for i, e in enumerate(mb):
            L = len(e["tokens"])
            input_ids[i, :L] = torch.tensor(e["tokens"], dtype=torch.long)
            labels[i, :L]    = torch.tensor(e["targets"], dtype=torch.long)
            attn_mask[i, :L] = 1

        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            out  = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels, use_cache=False)
            loss = out.loss / n_accum

        loss.backward()
        loss_sum += loss.item()

        del out, loss, input_ids, labels, attn_mask
        torch.cuda.empty_cache()

    # linear LR decay
    lr = LEARNING_RATE * (1 - step / num_steps)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    clip_grad_norm_(trainable, max_norm=1.0)
    optimizer.step()
    optimizer.zero_grad()

    step += 1
    peak_gb = torch.cuda.max_memory_allocated() / 1e9
    print(f"step {step}/{num_steps}: loss={loss_sum:.4f}, lr={lr:.2e}, peak={peak_gb:.1f}GB")

print(f"\nTraining complete. Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 6694 examples
Training: 209 steps, batch=32, micro=1, lr=0.0002
Unsloth: Will smartly offload gradients to save VRAM!
step 1/209: loss=11.7889, lr=2.00e-04, peak=69.9GB
step 2/209: loss=12.2489, lr=1.99e-04, peak=69.9GB
step 3/209: loss=10.0070, lr=1.98e-04, peak=69.9GB
step 4/209: loss=9.9354, lr=1.97e-04, peak=69.9GB
step 5/209: loss=7.7959, lr=1.96e-04, peak=69.9GB
step 6/209: loss=6.1199, lr=1.95e-04, peak=69.9GB
step 7/209: loss=5.8600, lr=1.94e-04, peak=69.9GB
step 8/209: loss=4.5199, lr=1.93e-04, peak=69.9GB
step 9/209: loss=4.2408, lr=1.92e-04, peak=69.9GB
step 10/209: loss=3.7129, lr=1.91e-04, peak=69.9GB
step 11/209: loss=3.7753, lr=1.90e-04, peak=69.9GB
step 12/209: loss=3.5421, lr=1.89e-04, peak=69.9GB
step 13/209: loss=3.3097, lr=1.89e-04, peak=69.9GB
step 14/209: loss=3.0064, lr=1.88e-04, peak=69.9GB
step 15/209: loss=2.8392, lr=1.87e-04, peak=69.9GB
step 16/209: loss=2.7731, lr=1.86e-04, peak=69.9GB
step 17/209: loss=2.6757, lr=1.85e-04, peak=69.9GB
step 18/209: l

In [8]:
# ── Cell 8: Save & package ─────────────────────────────────────────────────
import zipfile
from safetensors.torch import load_file, save_file

model.save_pretrained(SFT_OUT)
tokenizer.save_pretrained(SFT_OUT)

st_path = os.path.join(SFT_OUT, "adapter_model.safetensors")
if os.path.exists(st_path):
    tensors = load_file(st_path)
    renamed = {
        k.replace("base_model.model.lm_head.", "base_model.model.backbone.lm_head."): v
        for k, v in tensors.items()
    }
    save_file(renamed, st_path)

adapter_files = [f for f in os.listdir(SFT_OUT) if f.startswith("adapter")]
with zipfile.ZipFile(os.path.join(SFT_OUT, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in adapter_files:
        zf.write(os.path.join(SFT_OUT, fname), fname)

size = os.path.getsize(os.path.join(SFT_OUT, "submission.zip")) / 1e6
print(f"submission.zip ready: {size:.1f} MB")
print("Done.")

submission.zip ready: 54.8 MB
Done.
